# Qwen3.6-27B Agentic Assistant — Training Orchestrator

One notebook that runs the full Phase 4 sequence on the **RTX PRO 6000 box** (96GB VRAM / 180GB RAM / 200GB disk):

1. Environment sanity checks (GPU, disk, dependencies)
2. Pre-flight `--dry-run` — validates config, data, **loss masking**, and packing at **no GPU cost**
3. Stage 1: SFT LoRA (`train_sft.py`, Unsloth, bf16 base)
4. Merge adapter → bf16 checkpoint, reclaim disk (`merge_and_export.py`)
5. Stage 2: ORPO preference pass (`train_orpo.py`, TRL)
6. Final merge + sequential GGUF export with offload

**How this notebook works:** every step shells out to the versioned scripts in this repo — the notebook holds *no logic of its own*, so a crashed kernel loses nothing. Every training stage is resumable: re-running a training cell after a crash auto-resumes from the last integrity-checked checkpoint (`resume: auto` in the configs).

**Colab portability:** the scripts have no box-specific assumptions, but a 27B bf16-base SFT does not fit Colab's A100-40GB — you would need `configs/sft_qlora_fallback.yaml` with a reduced `max_seq_len`, and even then it is tight. Treat Colab as the emergency fallback, per the Phase 1 decision.

**Long-running cells vs. SSH:** for the multi-hour SFT stage, the more robust pattern is running the same command under `tmux` over SSH and using this notebook only for pre-flight and post-processing. Both work; the notebook prints the exact command either way.

## Step 0 — Environment sanity checks

Verifies the GPU, CUDA, disk headroom, and that the Phase 3 output (`data/train.jsonl`) is where the config expects it. Nothing is installed or modified here.

In [ ]:
import shutil, subprocess, sys
from pathlib import Path

REPO = Path.cwd()  # run this notebook from the training/ directory
assert (REPO / "train_sft.py").exists(), f"run this notebook from training/, not {REPO}"

# GPU check
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print("GPU :", gpu.stdout.strip() or "NOT FOUND — nvidia-smi failed")

# Disk check (Phase 1 budget: need ~30GB free just to start, much more for merge)
usage = shutil.disk_usage(REPO)
free_gb = usage.free / 1024**3
print(f"Disk: {free_gb:.1f} GiB free of {usage.total / 1024**3:.1f} GiB")
if free_gb < 30:
    print("WARNING: below the 30GB pre-training guard — clear space before training")

# Phase 3 output present?
train_jsonl = REPO / "data" / "train.jsonl"
print("Data:", f"{train_jsonl} ({train_jsonl.stat().st_size / 1024**2:.1f} MiB)"
      if train_jsonl.exists() else f"{train_jsonl} MISSING — run the Phase 3 pipeline first")

print("Py  :", sys.version.split()[0])

## Step 1 — Install dependencies

Unsloth is installed *first* and pinned by their own installer logic (it patches transformers, so version pairing matters). Everything else comes from `requirements.txt`. Re-running this cell is safe — installs are idempotent.

If Unsloth errors on the Qwen3.6 architecture at your exact version, that is the trigger to switch to the documented Axolotl fallback (see `README.md` § Fallback).

In [ ]:
%pip install --upgrade unsloth
%pip install -r requirements.txt

# Quick import smoke test — fail HERE, not 40 minutes into a run.
import importlib
for mod in ("unsloth", "trl", "peft", "datasets", "transformers", "yaml"):
    importlib.import_module(mod)
    print(f"ok: {mod}")

import torch
print(f"torch {torch.__version__}, cuda={torch.cuda.is_available()}, "
      f"bf16 supported={torch.cuda.is_bf16_supported() if torch.cuda.is_available() else 'n/a'}")

## Step 2 — Run the unit suite, then the pre-flight dry run

Two gates before any GPU time is spent:

1. **Unit suite** (60 tests): masking, packing, config validation, resume integrity, ORPO pair rendering. Seconds to run; catches code-level regressions.
2. **`--dry-run`**: the real thing — downloads only the **tokenizer**, then runs the mask-verification gate on *your actual* `data/train.jsonl`, tokenizes and packs the full corpus, and prints token counts, trainable fraction, packing efficiency and the step plan. If the mask check fails here, training would have failed silently — this is exactly the check that makes response-only masking trustworthy on a brand-new hybrid architecture.

**Read the dry-run output before proceeding.** Healthy signs: `mask check: PASS`, trainable fraction mean roughly 0.10–0.45 for an agentic-heavy mix, packing efficiency > 0.90, overlong-drop rate < 10%.

In [ ]:
!python -m pytest
!python train_sft.py --config configs/sft_plan_a.yaml --dry-run

## Step 3 — Stage 1: SFT LoRA (the long one)

Expected ~20–25 GPU-hours for the Plan A mix at 2 epochs. The run:

- checkpoints every 100 optimizer steps (~45–60 min apart), keeping the newest 3;
- guards free disk before every save;
- prints per-pattern LoRA target-module coverage right after model load — **check that no pattern matched 0 modules**; if the hybrid DeltaNet blocks expose extra projection names, add them to `lora.target_modules` in the config and restart (auto-resume makes the restart cheap);
- auto-resumes if you re-run this cell after any interruption.

For maximum robustness over SSH instead of the notebook, run the identical command under tmux:

```bash
tmux new -s sft
cd training && python train_sft.py --config configs/sft_plan_a.yaml 2>&1 | tee runs/sft_plan_a/train.log
```

If you hit an OOM at 16K sequences (possible — hybrid-arch kernel memory behavior is newer than the Qwen3.5 numbers the plan was based on), switch to `configs/sft_qlora_fallback.yaml`; it is the same run with a 4-bit base and 32K headroom.

In [ ]:
!python train_sft.py --config configs/sft_plan_a.yaml 2>&1 | tee -a runs_sft.log

## Step 4 — Merge SFT adapter → bf16 checkpoint, reclaim disk

The merge runs on **CPU/system RAM** (180GB is plenty; the GPU is not needed) and needs ~58GB of free disk *before* it starts — the script guards for that. After the merged output is **verified on disk**, `--delete-hf-cache-after` reclaims the ~55GB base-model download. Order matters: verify first, delete second; the script enforces it.

Restart the notebook kernel (or at minimum ensure the training model is out of VRAM/RAM) before running this if Step 3 ran in-process.

In [ ]:
!python merge_and_export.py \
    --config configs/sft_plan_a.yaml \
    --adapter runs/sft_plan_a/final_adapter \
    --output artifacts/merged_sft_bf16 \
    --delete-hf-cache-after

## Step 5 — Stage 2: ORPO preference pass

Runs TRL's `ORPOTrainer` on the **merged** SFT checkpoint with a fresh r=32 LoRA, 4-bit base (ORPO forwards chosen + rejected, roughly doubling activation memory). Needs `data/preference_pairs.jsonl` — the 5–15k pairs targeting plans-before-acting / correct-tool-schema / asks-clarifying-questions from Phase 2's specialization work.

The dry run validates and stats the pairs first; expect well under 20% rejected rows, or go inspect the pair generator.

In [ ]:
!python train_orpo.py --config configs/orpo_plan_a.yaml --dry-run
!python train_orpo.py --config configs/orpo_plan_a.yaml 2>&1 | tee -a runs_orpo.log

## Step 6 — Final merge + sequential GGUF export

Merges the ORPO adapter onto the merged-SFT checkpoint (note `--config configs/orpo_plan_a.yaml` — its `model.name_or_path` points at the merged SFT model, which is the correct merge base for this stage), then exports GGUFs **one at a time**: each quant is written, offloaded by your command, and deleted locally before the next starts, so peak disk usage is one f16 intermediate plus one quant.

Set `OFFLOAD` to your real destination (rclone remote, scp target, or `huggingface-cli upload`). To keep a quant locally instead, drop `--offload-cmd`.

Prerequisite: a built llama.cpp checkout (`cmake -B build && cmake --build build --config Release -j`).

In [ ]:
LLAMA_CPP = "~/llama.cpp"                      # <-- your llama.cpp checkout
OFFLOAD = "rclone move {path} remote:models/"  # <-- your offload destination

!python merge_and_export.py \
    --config configs/orpo_plan_a.yaml \
    --adapter runs/orpo_plan_a/final_adapter \
    --output artifacts/final_bf16 \
    --gguf q4_k_m q6_k \
    --llama-cpp-dir {LLAMA_CPP} \
    --offload-cmd "{OFFLOAD}"

## Step 7 — Quick smoke test of the final model

A minimal generation check with the merged bf16 model: one tool-calling prompt, one STEM prompt. This is *not* the evaluation suite (that is Phase 6) — it only confirms the artifact loads and produces well-formed ChatML with sane content before you spend time on downstream quantization testing.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "artifacts/final_bf16"   # or artifacts/merged_sft_bf16 to test pre-ORPO

tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)

prompts = [
    [{"role": "user", "content":
      "Plan, step by step, how you would debug a Docker container that "
      "exits immediately with code 137. Then give the first command you would run."}],
    [{"role": "user", "content":
      "A 2 kg mass hangs from a spring with k = 800 N/m. What is the "
      "natural frequency of oscillation in Hz?"}],
]

for messages in prompts:
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=400, temperature=0.7, do_sample=True)
    print("=" * 80)
    print(tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=False))